In [62]:
import optimize
import numpy as np
import pandas as pd
import importlib

conversion_to_annual = {
    "1d": 252,
    "1wk": 52,
    "1mo": 12,
}
annual_risk_free = 0.02

In [63]:
importlib.reload(optimize)

<module 'optimize' from '/Users/linanpluimgmail.com/repos/portfolio_optimizing/optimize.py'>

Current portfolio

In [64]:
# My current portfolio
tickers = [
            "VWCE.DE", #Vanguard FTSE All-World UCITS ETF (USD) Accumulating
            "IUSN.DE" #iShares MSCI World Small Cap UCITS ETF
           ]
weights = np.array([0.074, 0.926])  
interval = "1mo"  # "1d", "1wk", or "1mo"
anualization_factor = conversion_to_annual[interval]

risk_free = (1 + annual_risk_free) ** (1 / anualization_factor) - 1

returns, mu, cov = optimize.download_returns(
    tickers,
    interval,
)
print("min date:", returns.index.min().date())
print("max date:", returns.index.max().date())

stats, annual_mu, annual_cov = optimize.annualized_stats(
    returns,
    interval,
)
print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

current_gm = optimize.sample_geometric_mean(weights, returns)
current_sharpe = optimize.sharpe_ratio(weights, mu, cov, risk_free)

print(f'\nCurrent {interval} GM:', round(current_gm, 6))
print("Annualized GM:", round((1 + current_gm) ** anualization_factor - 1, 6))
print(f'Current {interval} Sharpe:', round(current_sharpe, 6))
print("Annualized Sharpe:", round(current_sharpe * np.sqrt(anualization_factor), 6))

[*********************100%***********************]  2 of 2 completed

min date: 2019-08-01
max date: 2026-08-01

Annualized ETF statistics:
Ticker             IUSN.DE  VWCE.DE
geometric_return    0.1013   0.1272
arithmetic_return   0.1121   0.1294
volatility          0.1742   0.1348

Current 1mo GM: 0.009903
Annualized GM: 0.125531
Current 1mo Sharpe: 0.228639
Annualized Sharpe: 0.792027


Optimization

In [65]:
# Settings for optimization
tickers = [
            "VWCE.DE", #Vanguard FTSE All-World UCITS ETF (USD) Accumulating
            "IUSN.DE", #iShares MSCI World Small Cap UCITS ETF
            "PPFB.DE", #iShares Physical Gold ETC
            "SXRS.DE", #iShares Diversified Commodity Swap UCITS ETF
            "SEC0.DE" #iShares MSCI Global Semiconductors UCITS ETF USD (Acc)
           ] 
interval = "1mo"  # "1d", "1wk", or "1mo"
start = None # or none if you want the whole history
end = None # or none if you want the whole history

In [71]:
# Optimize
returns, mu, cov = optimize.download_returns(
    tickers,
    interval,
    start=start,
    end=end,
)
print("min date:", returns.index.min().date())
print("max date:", returns.index.max().date())

stats, annual_mu, annual_cov = optimize.annualized_stats(
    returns,
    interval,
)
print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

anualization_factor = conversion_to_annual[interval]
risk_free = (1 + annual_risk_free) ** (1 / anualization_factor) - 1
sharpe = optimize.maximize_sharpe_ratio(
    returns,
    anualization_factor,
    risk_free,
)
gm = optimize.maximize_sample_geometric_mean(
    returns,
    anualization_factor,
    risk_free,
)

weights = pd.DataFrame(
    {
        "ETF": returns.columns,
        "Sharpe_weight": np.round(sharpe["weights"], 6),
        "GM_weight": np.round(gm["weights"], 6),
    }
).set_index("ETF")

print("\nPortfolio weights:")
print(weights.round(4))

print("\nAnnualized Sharpe Optimization Results:")
print(f"Optimal Sharpe:", round(sharpe["annual_sharpe"], 6))
print(f"Historical GM:", round(sharpe["annual_historical_gm"], 6))
print("\nAnnualized GM Optimization Results:")
print(f"Historical Sharpe:", round(gm["annual_historical_sharpe"], 6))
print(f"Optimal GM:", round(gm["annual_gm"], 6))



[*********************100%***********************]  5 of 5 completed


min date: 2021-09-01
max date: 2026-08-01

Annualized ETF statistics:
Ticker             IUSN.DE  PPFB.DE  SEC0.DE  SXRS.DE  VWCE.DE
geometric_return    0.0779   0.2068   0.3092   0.1177   0.1123
arithmetic_return   0.0872   0.2002   0.3368   0.1243   0.1148
volatility          0.1567   0.1487   0.3707   0.1609   0.1270

Portfolio weights:
         Sharpe_weight  GM_weight
ETF                              
IUSN.DE         0.0000        0.0
PPFB.DE         0.5554        0.0
SEC0.DE         0.1782        1.0
SXRS.DE         0.2663        0.0
VWCE.DE         0.0000        0.0

Annualized Sharpe Optimization Results:
Optimal Sharpe: 1.557333
Historical GM: 0.216514

Annualized GM Optimization Results:
Historical Sharpe: 0.855082
Optimal GM: 0.309239
